# Summary_Day13_offline.ipynb  
## 사전학습 Pretrained 모델 활용 1 · 인터넷 불가 버전 · FakeData / weights=None 구조 연습

이 파일은 **인터넷이 안 되는 환경**에서 13강 사전학습 모델 활용 흐름을 연습하기 위한 버전이다.

사전학습 모델을 진짜로 사용하려면 보통 ImageNet 가중치를 다운로드해야 한다.  
인터넷이 없으면 `weights=...DEFAULT` 또는 `pretrained=True`가 실패할 수 있다.

그래서 이 파일은 다음처럼 구성한다.

```text
데이터 다운로드 없음 → torchvision.datasets.FakeData 사용
가중치 다운로드 없음 → weights=None 사용
구조는 동일하게 연습 → fc / classifier 교체, freeze, 학습 루프, 저장
```

> 중요한 정리:  
> `weights=None`은 진짜 사전학습 효과를 보여주는 것은 아니다.  
> 다만 인터넷 없이도 ResNet, VGG, MobileNet의 구조 수정 방법을 연습할 수 있다.

## 1. 전체 실습 목적

이번 오프라인 실습의 목적은 다음이다.

1. 인터넷 없이 실행 가능한 이미지 분류 데이터셋을 만든다.
2. ResNet18 구조를 불러오되 pretrained weight 다운로드는 하지 않는다.
3. ResNet18의 `fc`를 10 class에 맞게 교체한다.
4. `requires_grad=False`로 feature extractor freeze 구조를 연습한다.
5. VGG19-BN과 MobileNetV2의 마지막 layer 위치를 확인한다.
6. 짧은 학습 루프와 평가 루프를 실행한다.
7. 모델 저장 흐름을 확인한다.

## 2. 라이브러리 준비

`torchvision.datasets.FakeData`는 인터넷 없이 가짜 이미지 데이터셋을 만들어 준다.  
데이터 내용은 랜덤이지만, shape과 label 구조가 이미지 분류 실습에 적합하다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, models

torch.manual_seed(123)
np.random.seed(123)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("device:", device)
print("torchvision:", torchvision.__version__)

## 3. 클래스와 전처리 준비

FakeData도 CIFAR-10처럼 10개 class를 가진다고 가정한다.

이미지 크기는 사전학습 모델 구조 연습을 위해 3×112×112로 둔다.

In [ ]:
classes = (
    "plane", "car", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
)

n_output = len(classes)

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

print("class 수:", n_output)

## 4. FakeData 데이터셋 만들기

### 함수 사용법

```python
datasets.FakeData(size=400, image_size=(3,112,112), num_classes=10, transform=transform)
```

- `size`: 전체 샘플 수다.
- `image_size`: 이미지 Tensor 구조다.
- `num_classes`: class 개수다.
- `transform`: 이미지 전처리다.

인터넷 다운로드 없이 바로 생성된다.

In [ ]:
train_dataset = datasets.FakeData(
    size=400,
    image_size=(3, 112, 112),
    num_classes=n_output,
    transform=transform
)

test_dataset = datasets.FakeData(
    size=100,
    image_size=(3, 112, 112),
    num_classes=n_output,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

images, labels = next(iter(train_loader))

print("images:", images.shape)
print("labels:", labels.shape)
print("첫 label:", labels[0].item())

## 5. 이미지 확인 함수

FakeData 이미지는 랜덤 이미지라 의미 있는 사물은 아니지만, image classification 코드 흐름은 동일하다.

In [ ]:
def denormalize_image(image, mean=imagenet_mean, std=imagenet_std):
    mean_t = torch.tensor(mean).view(3, 1, 1)
    std_t = torch.tensor(std).view(3, 1, 1)

    image = image.cpu() * std_t + mean_t
    image = torch.clamp(image, 0, 1)

    return image


def show_sample_images(loader, classes, num_images=8):
    images, labels = next(iter(loader))

    plt.figure(figsize=(12, 6))

    for i in range(num_images):
        plt.subplot(2, 4, i + 1)

        img = denormalize_image(images[i]).permute(1, 2, 0).numpy()

        plt.imshow(img)
        plt.title(classes[labels[i].item()])
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_sample_images(train_loader, classes, num_images=8)

## 6. AdaptiveAvgPool2d 구조 확인

오프라인에서도 사전학습 모델의 핵심 구조인 Adaptive Pooling은 그대로 확인할 수 있다.

In [ ]:
p = nn.AdaptiveAvgPool2d((1, 1))
l1 = nn.Linear(32, 10)

inputs = torch.randn(16, 32, 20, 20)

m1 = p(inputs)
m2 = m1.view(m1.shape[0], -1)
m3 = l1(m2)

print("inputs:", inputs.shape)
print("after AdaptiveAvgPool2d:", m1.shape)
print("after view:", m2.shape)
print("after Linear:", m3.shape)

## 7. ResNet18 구조 불러오기 weights=None

인터넷이 없으므로 사전학습 가중치를 다운로드하지 않는다.

### 함수 사용법

```python
models.resnet18(weights=None)
```

- 모델 구조만 만든다.
- ImageNet으로 학습된 weight는 없다.
- 구조 수정 연습용이다.

In [ ]:
net_resnet = models.resnet18(weights=None)

print("기존 fc:")
print(net_resnet.fc)
print("fc in_features:", net_resnet.fc.in_features)

## 8. ResNet18 fc 교체

CIFAR-10식 10 class에 맞게 마지막 layer를 교체한다.

In [ ]:
fc_in_features = net_resnet.fc.in_features
net_resnet.fc = nn.Linear(fc_in_features, n_output)

net_resnet = net_resnet.to(device)

print("교체된 fc:")
print(net_resnet.fc)

## 9. 짧은 학습/평가 함수

FakeData는 랜덤 데이터라 성능 자체는 의미가 크지 않다.  
여기서는 사전학습 모델 구조를 학습 루프에 연결하는 코드 흐름을 확인한다.

In [ ]:
def fit(model, train_loader, test_loader, criterion, optimizer, device, num_epochs=1):
    history = []

    for epoch in range(num_epochs):
        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            pred = torch.max(outputs, 1)[1]

            train_loss += loss.item() * labels.size(0)
            train_correct += (pred == labels).sum().item()
            train_total += labels.size(0)

        model.eval()

        test_correct = 0
        test_total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                pred = torch.max(outputs, 1)[1]

                test_correct += (pred == labels).sum().item()
                test_total += labels.size(0)

        history.append([
            epoch + 1,
            train_loss / train_total,
            train_correct / train_total,
            test_correct / test_total
        ])

        print(
            f"epoch {epoch + 1} | "
            f"train_loss={history[-1][1]:.4f} | "
            f"train_acc={history[-1][2]:.4f} | "
            f"test_acc={history[-1][3]:.4f}"
        )

    return np.array(history)

## 10. ResNet18 구조 연습 학습

랜덤 데이터라 정확도 향상보다 코드 흐름을 보는 것이 목적이다.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net_resnet.parameters(), lr=0.001, momentum=0.9)

history_resnet = fit(
    net_resnet,
    train_loader,
    test_loader,
    criterion,
    optimizer,
    device,
    num_epochs=1
)

## 11. Transfer Learning freeze 구조 연습

실제 전이학습에서는 사전학습된 feature extractor를 고정하고 마지막 layer만 학습한다.  
오프라인에서는 weight가 pretrained가 아니므로 효과는 다르지만, 코드 구조는 그대로 연습할 수 있다.

In [ ]:
net_transfer = models.resnet18(weights=None)

for param in net_transfer.parameters():
    param.requires_grad = False

net_transfer.fc = nn.Linear(net_transfer.fc.in_features, n_output)

net_transfer = net_transfer.to(device)

trainable_params = sum(p.numel() for p in net_transfer.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in net_transfer.parameters())

print("total params:", total_params)
print("trainable params:", trainable_params)
print("학습 대상은 새 fc layer 중심이다.")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net_transfer.fc.parameters(), lr=0.001, momentum=0.9)

history_transfer = fit(
    net_transfer,
    train_loader,
    test_loader,
    criterion,
    optimizer,
    device,
    num_epochs=1
)

## 12. VGG19-BN 마지막 layer 위치 확인

VGG19-BN의 마지막 Linear layer는 `classifier[6]`이다.

오프라인에서는 구조만 확인하고, 무거운 모델이라 기본 실행은 선택형으로 둔다.

In [ ]:
RUN_VGG = False

if RUN_VGG:
    net_vgg = models.vgg19_bn(weights=None)

    print("기존 classifier[6]:")
    print(net_vgg.classifier[6])

    in_features = net_vgg.classifier[6].in_features
    net_vgg.classifier[6] = nn.Linear(in_features, n_output)

    net_vgg.features = net_vgg.features[:-1]
    net_vgg.avgpool = nn.Identity()

    print("교체 후 classifier[6]:")
    print(net_vgg.classifier[6])
else:
    print("VGG19-BN은 무거워서 기본 실행에서는 건너뛴다.")
    print("실행하려면 RUN_VGG = True로 바꾼다.")

## 13. MobileNetV2 classifier 교체

MobileNetV2의 마지막 Linear layer는 `classifier[1]`이다.

오프라인에서는 `weights=None`으로 구조를 만들고 마지막 layer를 교체한다.

In [ ]:
model_mobile = models.mobilenet_v2(weights=None)

print("기존 classifier:")
print(model_mobile.classifier)

num_features = model_mobile.classifier[1].in_features
model_mobile.classifier[1] = nn.Linear(num_features, n_output)

model_mobile = model_mobile.to(device)

print("\n교체 후 classifier:")
print(model_mobile.classifier)

## 14. MobileNetV2 짧은 학습과 저장

구조 교체 후 학습 루프와 저장 흐름을 확인한다.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_mobile.parameters(), lr=0.001, momentum=0.9)

history_mobile = fit(
    model_mobile,
    train_loader,
    test_loader,
    criterion,
    optimizer,
    device,
    num_epochs=1
)

model_path = "/mnt/data/day13_offline_mobilenetv2_state_dict.pth"

torch.save(model_mobile.state_dict(), model_path)

print("모델 저장 완료:", model_path)

## 15. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `weights=None` | 사전학습 가중치 없이 구조만 생성 | 인터넷 불가 구조 연습 |
| `FakeData` | 가짜 이미지 데이터셋 | 다운로드 없이 실행 |
| `fc` | ResNet 마지막 분류 layer | `net.fc` |
| `classifier[6]` | VGG 마지막 Linear | `net.classifier[6]` |
| `classifier[1]` | MobileNetV2 마지막 Linear | `model.classifier[1]` |
| `requires_grad` | 학습 여부 | `False`면 freeze |
| `state_dict` | 모델 파라미터 | 저장/불러오기 |
| `AdaptiveAvgPool2d` | 크기 고정 풀링 | `[N,C,H,W] → [N,C,1,1]` |

## 16. 시험용 요약

```text
인터넷 불가 버전 = 사전학습 가중치 다운로드 없이 구조 수정 방법을 연습한다
```

꼭 기억할 것:

- 진짜 pretrained 효과를 보려면 ImageNet weight가 필요하다.
- 인터넷이 없으면 `weights=None`으로 구조만 만들 수 있다.
- `weights=None`은 사전학습 가중치를 사용하지 않는다는 뜻이다.
- ResNet18의 마지막 layer는 `fc`다.
- VGG19-BN의 마지막 layer는 `classifier[6]`이다.
- MobileNetV2의 마지막 layer는 `classifier[1]`이다.
- Transfer Learning은 앞쪽 파라미터를 freeze하고 마지막 layer만 학습하는 구조다.
- `requires_grad=False`는 해당 파라미터를 업데이트하지 않는다는 뜻이다.
- FakeData는 인터넷 없이 이미지 분류 코드 흐름을 테스트할 때 유용하다.